# Anomaly Detection Monitoring

* This notebook demonstrates how to use the deployed model for anomaly detection and for pushing metrics to the OCI Monitoring service.
* This notebook is intended for demonstration purposes only. The production-ready implementation is also available in `job_folder/monitoring.py`, which is used by `job_automation.py` for automated monitoring workflows.
* The notebook covers the following topics:
  * Importing weekly data from OCI Object Storage
  * Processing and aggregating the data to align with the business problem and model requirements
  * Using the deployed model to generate predictions and prediction intervals
  * Flagging anomalies, defined as observations that fall outside the prediction interval
  * Pushing metrics to OCI Monitoring

Conda Environment: `generalml_p311_cpu_x86_64_v1`

Author: Assaf Rabinowicz, Data Scientist EMEA

# Package and Data Collection

In [1]:
import io
import os
import pandas as pd
import requests
import oci
import sys
import pandas as pd

from job_folder import helper
from job_folder import config

In [2]:
bucket_name = config.BUCKET_NAME
folder_name= config.FOLDER_NAME
feature_cols = config.FEATURE_COLS
date_col= config.DATE_COL
target_col= config.TARGET_COL
compartment_id=config.COMPARTMENT
monitoring_namespace=config.MONITORING_NAMESPACE
metric_name=config.MONITORING_METRIC_NAME
endpoint=config.ENDPOINT

signer = oci.auth.signers.get_resource_principals_signer()

In [3]:
input_file_name='2012-08-03.csv' # relevant only for this notebook. The file name will be sent as environment variable in the job.

In [15]:
object_storage_client = oci.object_storage.ObjectStorageClient(
    config={},
    signer=signer
)

namespace_name = object_storage_client.get_namespace().data

In [16]:
df = helper.read_csv_from_object_storage(
    client=object_storage_client,
    namespace=namespace_name,
    bucket=bucket_name,
    folder_name=folder_name,
    object_name=input_file_name
)
df.head()

,Unnamed: 0,Store,Date,IsHoliday,Dept,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,321201,33,2012-08-03,0,52.0,9.00,92.13,3.595,238.26,0.00,2.88,44.29,767.86,130.737871,7.147,3,39690
1,147727,15,2012-08-03,0,27.0,1262.94,73.13,3.819,7251.41,216.62,0.48,4906.29,1420.37,138.173581,8.193,2,123737
2,147728,15,2012-08-03,0,81.0,4051.58,73.13,3.819,7251.41,216.62,0.48,4906.29,1420.37,138.173581,8.193,2,123737
3,147729,15,2012-08-03,0,4.0,14832.84,73.13,3.819,7251.41,216.62,0.48,4906.29,1420.37,138.173581,8.193,2,123737
4,321192,33,2012-08-03,0,5.0,35.00,92.13,3.595,238.26,0.00,2.88,44.29,767.86,130.737871,7.147,3,39690


# Date Processing

In [6]:
agg_df = helper.create_aggregated_df(df)

if len(agg_df) != 1:
    raise ValueError(f"Expected exactly 1 aggregated row, got {len(agg_df)}")
features_df = agg_df[feature_cols].copy()
features_df.head()

,Dept,Temperature,Fuel_Price,CPI,Unemployment,Type,Size
0,2967,79.944317,3.568967,175.157356,7.232204,2.402764,136191.124031


In [7]:
features_df = features_df[feature_cols].copy()
payload = features_df.to_dict(orient="list")

# Scoring and Metrics Collection

In [8]:
result = helper.call_deployed_model(endpoint, signer, payload)

In [9]:
prediction = float(result["prediction"])
lower = float(result["lower_prediction_interval"])
upper = float(result["upper_prediction_interval"])
actual = float(agg_df[target_col].iloc[0])

is_anomaly = actual < lower or actual > upper
print(is_anomaly)

False


# Monitoring

In [10]:
monitoring_client = oci.monitoring.MonitoringClient(
    config={},
    signer=signer,
    service_endpoint="https://telemetry-ingestion.eu-frankfurt-1.oraclecloud.com"
)

In [11]:
actual

47485899.56

In [18]:
response = helper.push_forecast_metrics(
    monitoring_client=monitoring_client,
    compartment_id=compartment_id,
    namespace=monitoring_namespace,
    metric_name=metric_name,
    actual=actual,
    predicted=prediction,
    lower=lower,
    upper=upper,
)
print(response.data)

{
  "failed_metrics": [],
  "failed_metrics_count": 0
}
